# 04 — Fair stress tests: fairness, robustness, masking

Cache-first audit notebook for the locked roster. Prefers:
- `data/processed/fairness_results.json`
- `data/processed/robustness_results.json`
- `data/processed/masking_results.json`

Optional small live probes via `LIVE_RERUN = True`.

CLIs: `run_fairness.py`, `run_robustness.py`, `run_masking.py`


In [ ]:
%matplotlib inline
import json, sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Walk up until DeepProjectHate/src/paths.py is found (fixes wrong-cwd PathLike/None errors).
_src = None
for _p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _cand = _p / "src"
    if (_cand / "paths.py").is_file():
        _src = str(_cand)
        break
if _src is None:
    raise FileNotFoundError(f"Cannot find DeepProjectHate/src/paths.py from cwd={Path.cwd()}")
if _src not in sys.path:
    sys.path.insert(0, _src)

for _mod in ("paths", "roster", "model_registry", "baselines", "splits", "device"):
    sys.modules.pop(_mod, None)

from paths import PROCESSED_DIR, PROJECT_ROOT, SEED
from roster import FAIR_ROSTER, print_roster_status

np.random.seed(SEED)
sns.set_theme(style="whitegrid")
print(f"Project root: {PROJECT_ROOT}")
print(f"src: {_src}")
print("Roster:")
READY = print_roster_status(FAIR_ROSTER)

LIVE_RERUN = False


## 1. Fairness (per-identity FPR)


In [ ]:
FAIRNESS = PROCESSED_DIR / "fairness_results.json"
if FAIRNESS.exists():
    payload = json.loads(FAIRNESS.read_text(encoding="utf-8"))
    rows = []
    for key in FAIR_ROSTER:
        block = payload.get("models", {}).get(key)
        if not block:
            print(f"  [CACHE MISS] fairness/{key}")
            continue
        per_group = block.get("per_group") or block.get("groups") or []
        max_fpr = None
        if isinstance(per_group, list) and per_group:
            max_fpr = max(g.get("fpr", 0) for g in per_group)
        rows.append({
            "model": key,
            "overall_fpr": block.get("overall_fpr"),
            "control_fpr": block.get("control_fpr"),
            "max_group_fpr": max_fpr if max_fpr is not None else block.get("max_group_fpr"),
        })
    fairness_df = pd.DataFrame(rows)
    display(fairness_df.round(3))
    print(f"Loaded {FAIRNESS}")
    if not fairness_df.empty and "overall_fpr" in fairness_df:
        ax = sns.barplot(data=fairness_df, x="model", y="overall_fpr", color="crimson")
        ax.set_title("Overall FPR on identity probe")
        ax.tick_params(axis="x", rotation=30)
        plt.tight_layout(); plt.show()
else:
    print(f"Missing {FAIRNESS}. Run: python scripts/run_fairness.py")
    fairness_df = pd.DataFrame()


## 2. Robustness (perturbation evasion)


In [ ]:
ROBUST = PROCESSED_DIR / "robustness_results.json"
if ROBUST.exists():
    payload = json.loads(ROBUST.read_text(encoding="utf-8"))
    rows = []
    models = payload.get("models", payload)
    for key in FAIR_ROSTER:
        block = models.get(key) if isinstance(models, dict) else None
        if not block:
            print(f"  [CACHE MISS] robustness/{key}")
            continue
        if isinstance(block, dict) and "perturbations" in block:
            for pname, pblock in block["perturbations"].items():
                rows.append({
                    "model": key,
                    "perturbation": pname,
                    **{k: pblock.get(k) for k in ("evasion_rate", "mean_drop", "f1", "accuracy") if isinstance(pblock, dict)},
                })
        else:
            rows.append({
                "model": key,
                **{k: block.get(k) for k in ("evasion_rate", "mean_drop") if isinstance(block, dict)},
            })
    robust_df = pd.DataFrame(rows)
    display(robust_df.round(3))
    print(f"Loaded {ROBUST}")
    if not robust_df.empty and "evasion_rate" in robust_df.columns:
        if "perturbation" in robust_df.columns:
            g = sns.catplot(
                data=robust_df, x="model", y="evasion_rate", hue="perturbation",
                kind="bar", height=4, aspect=1.6,
            )
            g.set_xticklabels(rotation=30)
            g.fig.suptitle("Evasion rate by perturbation", y=1.02)
            plt.show()
        else:
            ax = sns.barplot(data=robust_df, x="model", y="evasion_rate", color="darkorange")
            ax.set_title("Evasion rate"); ax.tick_params(axis="x", rotation=30)
            plt.tight_layout(); plt.show()
else:
    print(f"Missing {ROBUST}. Run: python scripts/run_robustness.py --quick")
    robust_df = pd.DataFrame()


## 3. Masking intervention


In [ ]:
MASK = PROCESSED_DIR / "masking_results.json"
if MASK.exists():
    payload = json.loads(MASK.read_text(encoding="utf-8"))
    ablation = payload.get("inference_ablation", payload.get("models", {}))
    rows = []
    for key in FAIR_ROSTER:
        block = ablation.get(key) if isinstance(ablation, dict) else None
        if not block:
            print(f"  [CACHE MISS] masking/{key}")
            continue
        if isinstance(block, dict):
            rows.append({"model": key, **block})
    mask_df = pd.DataFrame(rows)
    display(mask_df.round(3) if not mask_df.empty else mask_df)
    print(f"Loaded {MASK}")
    if "svm_training_masking" in payload:
        print("SVM training-masking:")
        display(pd.DataFrame([payload["svm_training_masking"]]).round(3))
    cols = [c for c in ("fpr_with_identity", "fpr_masked", "fpr_before", "fpr_after") if c in mask_df.columns]
    if len(cols) >= 2:
        plot_df = mask_df.melt(id_vars=["model"], value_vars=cols[:2], var_name="stage", value_name="fpr")
        ax = sns.barplot(data=plot_df, x="model", y="fpr", hue="stage")
        ax.set_title("Masking intervention FPR"); ax.tick_params(axis="x", rotation=30)
        plt.tight_layout(); plt.show()
else:
    print(f"Missing {MASK}. Run: python scripts/run_masking.py")
    mask_df = pd.DataFrame()


## 4. Optional live probes


In [ ]:
if not LIVE_RERUN:
    print("LIVE_RERUN=False — skipping live stress probes (cache-first).")
else:
    from fairness import build_identity_probe_df, per_group_fpr
    from model_registry import ALL_SPECS, load_scorer
    from splits import get_split_frame

    probe = build_identity_probe_df()
    for key in READY[:3]:
        try:
            scorer = load_scorer(ALL_SPECS[key])
            if key == "tfidf-svm":
                train = get_split_frame("train")
                scorer.fit(train["comment_text"].tolist(), train["toxic"].values)
            p = scorer.predict_toxic_proba(probe["text"].tolist())
            g = per_group_fpr(probe, p)
            print(f"\n{key} top-FPR groups:")
            display(g.sort_values("fpr", ascending=False).head(5).round(3))
        except Exception as exc:
            print(f"  [SKIP live] {key}: {exc}")
